# Segundo modelo — predicción de ronda de draft

En este notebook exploro mejoras sobre el modelo base (`xgb_draft_balanceado.pkl`) con tres estrategias:

1. **RandomizedSearchCV** sobre XGBoost para encontrar mejores hiperparámetros
2. **LightGBM** como modelo alternativo con pesos balanceados
3. **Selección de variables** usando feature importance para quedarnos con las más relevantes

El objetivo es mejorar el macro avg F1, especialmente en R1 y R2 que son las clases que más nos interesan.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import os
from sklearn.model_selection import train_test_split, RandomizedSearchCV, StratifiedKFold, cross_val_score
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay, roc_auc_score, roc_curve
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.preprocessing import label_binarize
from xgboost import XGBClassifier
import lightgbm as lgb


## Cargo los datos y preparo X e y

Mismo preprocesado que en el primer modelo.

In [ ]:
ncaa = pd.read_csv('../../datos/procesados/ncaa_final.csv')

# separo variables y target
X = ncaa.drop(columns=['ronda', 'rango_pick'])
y = ncaa['ronda']

# one hot encoding de posicion
X = pd.get_dummies(X, columns=['posicion'], drop_first=False)

# encodifico el target
le_target_draft = LabelEncoder()
y_enc = le_target_draft.fit_transform(y)  # ND=0, R1=1, R2=2

print("Clases:", le_target_draft.classes_)
print("Distribución:", pd.Series(y_enc).value_counts(normalize=True).round(3).to_dict())


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y_enc,
    test_size=0.15,
    random_state=11,
    stratify=y_enc
)

print(f"Train: {X_train.shape} | Test: {X_test.shape}")


## Estrategia 1 — RandomizedSearchCV sobre XGBoost

El modelo base usaba parámetros fijos (`n_estimators=100`, `max_depth=4`, `learning_rate=0.1`). Aquí exploramos un espacio más amplio usando búsqueda aleatoria con validación cruzada estratificada para no perder el equilibrio de clases en cada fold.

In [ ]:
# calculo pesos balanceados para el entrenamiento
pesos_train = compute_sample_weight(class_weight='balanced', y=y_train)

# espacio de hiperparámetros a explorar
param_dist = {
    'n_estimators':   [100, 200, 300, 400],
    'max_depth':      [3, 4, 5, 6],
    'learning_rate':  [0.01, 0.05, 0.1, 0.2],
    'subsample':      [0.6, 0.7, 0.8, 1.0],
    'colsample_bytree': [0.6, 0.7, 0.8, 1.0],
    'min_child_weight': [1, 3, 5],
    'gamma':          [0, 0.1, 0.3, 0.5]
}

# modelo base sobre el que se busca
xgb_base = XGBClassifier(
    objective='multi:softmax',
    num_class=3,
    random_state=11,
    eval_metric='mlogloss'
)

# validación cruzada estratificada — 5 folds
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=11)

# búsqueda aleatoria: 50 combinaciones, puntuando por macro F1
rnd_search = RandomizedSearchCV(
    estimator=xgb_base,
    param_distributions=param_dist,
    n_iter=50,          # número de combinaciones a probar
    scoring='f1_macro', # optimizo macro F1 (penaliza por igual R1, R2 y ND)
    cv=cv,
    random_state=11,
    n_jobs=-1,          # uso todos los cores disponibles
    verbose=1
)

# entreno con pesos para que el modelo vea R1 y R2 amplificadas
rnd_search.fit(X_train, y_train, sample_weight=pesos_train)

print("\nMejores hiperparámetros encontrados:")
for k, v in rnd_search.best_params_.items():
    print(f"  {k}: {v}")
print(f"\nMejor F1 macro en CV: {rnd_search.best_score_:.4f}")


In [ ]:
# evalúo el mejor modelo en el test
mejor_xgb = rnd_search.best_estimator_

y_pred_rxgb = mejor_xgb.predict(X_test)

print("=== Mejor XGBoost (RandomizedSearch) ===")
print(classification_report(y_test, y_pred_rxgb, target_names=le_target_draft.classes_))


In [ ]:
# visualización del classification report
clases = le_target_draft.classes_.tolist()
report = classification_report(y_test, y_pred_rxgb, target_names=clases, output_dict=True)

precision = [report[c]['precision'] for c in clases]
recall    = [report[c]['recall']    for c in clases]
f1        = [report[c]['f1-score']  for c in clases]

metricas = {'precision': precision, 'recall': recall, 'f1-score': f1}
colores  = ['#3B6D11', '#A32D2D', '#854F0B']

x = np.arange(len(clases))
ancho = 0.25

fig, ax = plt.subplots(figsize=(9, 5))
fig.patch.set_facecolor('#0a1019')
ax.set_facecolor('#0a1019')

for i, (nombre, valores) in enumerate(metricas.items()):
    barras = ax.bar(x + i * ancho, valores, ancho, label=nombre, color=colores[i], alpha=0.85)
    for barra in barras:
        alto = barra.get_height()
        ax.text(barra.get_x() + barra.get_width() / 2, alto + 0.015,
                f'{alto:.2f}', ha='center', va='bottom', fontsize=9, color='white')

macro_f1 = report['macro avg']['f1-score']
ax.axhline(y=macro_f1, color='#f97316', linestyle='--', linewidth=1.2,
           label=f'macro avg f1 ({macro_f1:.2f})')

ax.set_xticks(x + ancho)
ax.set_xticklabels(clases, color='white', fontsize=12)
ax.set_ylim(0, 1.1)
ax.set_ylabel('valor', color='white')
ax.set_title('XGBoost + RandomizedSearch — métricas por clase', color='white', fontsize=13, pad=12)
ax.tick_params(colors='white')
ax.legend(facecolor='#1a2535', labelcolor='white', fontsize=9)
for spine in ax.spines.values():
    spine.set_edgecolor('#ffffff22')

plt.tight_layout()
plt.show()


In [ ]:
# matriz de confusión
cm_rxgb = confusion_matrix(y_test, y_pred_rxgb)
disp = ConfusionMatrixDisplay(cm_rxgb, display_labels=le_target_draft.classes_)
disp.plot(cmap='Blues')
plt.title('Matriz de Confusión — XGBoost + RandomizedSearch')
plt.tight_layout()
plt.show()


## Estrategia 2 — LightGBM con pesos balanceados

LightGBM es un boosting alternativo a XGBoost. Su ventaja principal es la velocidad (usa histogramas en vez de ordenar todos los datos en cada split), pero también suele generalizar mejor en datasets pequeños. Aquí uso `class_weight='balanced'` directamente en el modelo, que es más limpio que `compute_sample_weight`.

In [ ]:
lgbm_model = lgb.LGBMClassifier(
    objective='multiclass',
    num_class=3,
    n_estimators=300,
    max_depth=5,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    min_child_samples=10,  # mínimo de muestras en una hoja — regulariza el modelo
    class_weight='balanced',
    random_state=11,
    verbose=-1             # silencio los logs de entrenamiento
)

lgbm_model.fit(X_train, y_train)

y_pred_lgbm = lgbm_model.predict(X_test)

print("=== LightGBM balanceado ===")
print(classification_report(y_test, y_pred_lgbm, target_names=le_target_draft.classes_))


In [ ]:
# visualización LightGBM
report_lgbm = classification_report(y_test, y_pred_lgbm, target_names=clases, output_dict=True)

precision_l = [report_lgbm[c]['precision'] for c in clases]
recall_l    = [report_lgbm[c]['recall']    for c in clases]
f1_l        = [report_lgbm[c]['f1-score']  for c in clases]

metricas_l = {'precision': precision_l, 'recall': recall_l, 'f1-score': f1_l}

fig, ax = plt.subplots(figsize=(9, 5))
fig.patch.set_facecolor('#0a1019')
ax.set_facecolor('#0a1019')

for i, (nombre, valores) in enumerate(metricas_l.items()):
    barras = ax.bar(x + i * ancho, valores, ancho, label=nombre, color=colores[i], alpha=0.85)
    for barra in barras:
        alto = barra.get_height()
        ax.text(barra.get_x() + barra.get_width() / 2, alto + 0.015,
                f'{alto:.2f}', ha='center', va='bottom', fontsize=9, color='white')

macro_f1_l = report_lgbm['macro avg']['f1-score']
ax.axhline(y=macro_f1_l, color='#f97316', linestyle='--', linewidth=1.2,
           label=f'macro avg f1 ({macro_f1_l:.2f})')

ax.set_xticks(x + ancho)
ax.set_xticklabels(clases, color='white', fontsize=12)
ax.set_ylim(0, 1.1)
ax.set_ylabel('valor', color='white')
ax.set_title('LightGBM balanceado — métricas por clase', color='white', fontsize=13, pad=12)
ax.tick_params(colors='white')
ax.legend(facecolor='#1a2535', labelcolor='white', fontsize=9)
for spine in ax.spines.values():
    spine.set_edgecolor('#ffffff22')
plt.tight_layout()
plt.show()


In [ ]:
cm_lgbm = confusion_matrix(y_test, y_pred_lgbm)
disp_lgbm = ConfusionMatrixDisplay(cm_lgbm, display_labels=le_target_draft.classes_)
disp_lgbm.plot(cmap='Blues')
plt.title('Matriz de Confusión — LightGBM')
plt.tight_layout()
plt.show()


## Estrategia 3 — Feature importance + selección de variables

Uno de los problemas del modelo base es que tiene ~35 variables, muchas correlacionadas entre sí. Usando la importancia de variables del XGBoost optimizado, nos quedamos solo con las top N y vemos si el modelo mantiene o mejora el rendimiento con menos ruido.

In [ ]:
# importancia de variables del mejor XGBoost
importancias = pd.Series(
    mejor_xgb.feature_importances_,
    index=X_train.columns
).sort_values(ascending=False)

# visualización
fig, ax = plt.subplots(figsize=(10, 6))
fig.patch.set_facecolor('#0a1019')
ax.set_facecolor('#0a1019')
top20 = importancias.head(20)
ax.barh(top20.index[::-1], top20.values[::-1], color='#3B6D11', alpha=0.85)
ax.set_xlabel('importancia', color='white')
ax.set_title('Top 20 variables — XGBoost optimizado (ronda)', color='white', fontsize=13)
ax.tick_params(colors='white')
for spine in ax.spines.values():
    spine.set_edgecolor('#ffffff22')
plt.tight_layout()
plt.show()


In [ ]:
# me quedo con las top 15 variables más importantes
top15_vars = importancias.head(15).index.tolist()
print("Variables seleccionadas:")
for v in top15_vars:
    print(f"  - {v}")

X_train_top = X_train[top15_vars]
X_test_top  = X_test[top15_vars]


In [ ]:
# entreno el LightGBM con solo las top 15 variables
lgbm_top = lgb.LGBMClassifier(
    objective='multiclass',
    num_class=3,
    n_estimators=300,
    max_depth=5,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    min_child_samples=10,
    class_weight='balanced',
    random_state=11,
    verbose=-1
)

lgbm_top.fit(X_train_top, y_train)
y_pred_top = lgbm_top.predict(X_test_top)

print("=== LightGBM con top 15 variables ===")
print(classification_report(y_test, y_pred_top, target_names=le_target_draft.classes_))


## Comparativa final de modelos

Resumen de los tres enfoques contra el modelo base del primer notebook.

In [ ]:
# cargo predicciones del modelo base para comparar (lo reconstruyo rápido)
modelo_base_ref = XGBClassifier(
    objective='multi:softmax', num_class=3,
    n_estimators=100, max_depth=4, learning_rate=0.1,
    random_state=11, eval_metric='mlogloss'
)
pesos_base = compute_sample_weight(class_weight='balanced', y=y_train)
modelo_base_ref.fit(X_train, y_train, sample_weight=pesos_base)
y_pred_base = modelo_base_ref.predict(X_test)
rep_base = classification_report(y_test, y_pred_base, target_names=clases, output_dict=True)

# recojo macro F1 de cada modelo
modelos_nombre = ['Base (XGB)', 'XGB + RndSearch', 'LightGBM', 'LightGBM top15']
preds_lista    = [y_pred_base, y_pred_rxgb, y_pred_lgbm, y_pred_top]

resultados = []
for nombre, preds in zip(modelos_nombre, preds_lista):
    if nombre == 'LightGBM top15':
        rep = classification_report(y_test, preds, target_names=clases, output_dict=True)
    else:
        rep = classification_report(y_test, preds, target_names=clases, output_dict=True)
    resultados.append({
        'modelo': nombre,
        'macro_f1': rep['macro avg']['f1-score'],
        'f1_ND':    rep['ND']['f1-score'],
        'f1_R1':    rep['R1']['f1-score'],
        'f1_R2':    rep['R2']['f1-score'],
        'auc_macro': roc_auc_score(
            label_binarize(y_test, classes=[0, 1, 2]),
            modelo_base_ref.predict_proba(X_test) if nombre == 'Base (XGB)'
            else (mejor_xgb.predict_proba(X_test) if nombre == 'XGB + RndSearch'
                  else (lgbm_model.predict_proba(X_test) if nombre == 'LightGBM'
                        else lgbm_top.predict_proba(X_test_top))),
            multi_class='ovr', average='macro'
        )
    })

df_res = pd.DataFrame(resultados)
print(df_res.to_string(index=False))


In [ ]:
# visualización comparativa
fig, ax = plt.subplots(figsize=(10, 5))
fig.patch.set_facecolor('#0a1019')
ax.set_facecolor('#0a1019')

x_pos = np.arange(len(modelos_nombre))
cols_metricas = ['macro_f1', 'f1_R1', 'f1_R2']
colores_comp  = ['#f97316', '#A32D2D', '#854F0B']
etiquetas     = ['macro F1', 'F1 R1', 'F1 R2']
ancho_c = 0.22

for i, (col, color, etiq) in enumerate(zip(cols_metricas, colores_comp, etiquetas)):
    vals = [df_res.loc[j, col] for j in range(len(modelos_nombre))]
    barras = ax.bar(x_pos + i * ancho_c, vals, ancho_c, label=etiq, color=color, alpha=0.85)
    for b in barras:
        ax.text(b.get_x() + b.get_width() / 2, b.get_height() + 0.01,
                f'{b.get_height():.2f}', ha='center', va='bottom', fontsize=8, color='white')

ax.set_xticks(x_pos + ancho_c)
ax.set_xticklabels(modelos_nombre, color='white', fontsize=10)
ax.set_ylim(0, 1.0)
ax.set_ylabel('valor', color='white')
ax.set_title('Comparativa de modelos — predicción de ronda', color='white', fontsize=13, pad=12)
ax.tick_params(colors='white')
ax.legend(facecolor='#1a2535', labelcolor='white', fontsize=9)
for spine in ax.spines.values():
    spine.set_edgecolor('#ffffff22')
plt.tight_layout()
plt.show()


## Guardo el mejor modelo

Escojo el modelo con mejor macro F1 para guardarlo como segundo modelo. Si el RandomizedSearch no mejora significativamente, guardamos el LightGBM como alternativa para el Streamlit.

## Estrategia 4 — Random Forest con pesos balanceados

Random Forest es un ensemble de árboles de decisión independientes (bagging), a diferencia de XGBoost y LightGBM que construyen árboles en secuencia (boosting). Aquí `class_weight='balanced'` ajusta automáticamente el peso de cada clase inversamente proporcional a su frecuencia.

In [ ]:
from sklearn.ensemble import RandomForestClassifier

rf_model = RandomForestClassifier(
    n_estimators=300,
    max_depth=8,
    min_samples_leaf=5,
    class_weight='balanced',
    random_state=11,
    n_jobs=-1
)

rf_model.fit(X_train, y_train)
y_pred_rf = rf_model.predict(X_test)

print("=== Random Forest balanceado ===")
print(classification_report(y_test, y_pred_rf, target_names=le_target_draft.classes_))


In [ ]:
cm_rf = confusion_matrix(y_test, y_pred_rf)
disp_rf = ConfusionMatrixDisplay(cm_rf, display_labels=le_target_draft.classes_)
disp_rf.plot(cmap='Blues')
plt.title('Matriz de Confusión — Random Forest')
plt.tight_layout()
plt.show()


## Estrategia 5 — SVM con kernel RBF

SVM (Support Vector Machine) con kernel RBF busca el hiperplano de máximo margen en un espacio de alta dimensión. Es el modelo más diferente conceptualmente a los anteriores: en vez de construir árboles, separa clases mediante fronteras de decisión no lineales. Requiere escalar las variables porque es sensible a la magnitud.

In [ ]:
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

# pipeline: escala + SVM — el escalado es obligatorio para SVM
svm_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('svm', SVC(
        kernel='rbf',
        C=1.0,           # penalización por error de clasificación
        gamma='scale',   # gamma = 1 / (n_features * X.var())
        class_weight='balanced',
        probability=True, # necesario para predict_proba y AUC-ROC
        random_state=11
    ))
])

svm_pipeline.fit(X_train, y_train)
y_pred_svm = svm_pipeline.predict(X_test)

print("=== SVM kernel RBF ===")
print(classification_report(y_test, y_pred_svm, target_names=le_target_draft.classes_))


In [ ]:
cm_svm = confusion_matrix(y_test, y_pred_svm)
disp_svm = ConfusionMatrixDisplay(cm_svm, display_labels=le_target_draft.classes_)
disp_svm.plot(cmap='Blues')
plt.title('Matriz de Confusión — SVM RBF')
plt.tight_layout()
plt.show()


## Estrategia 6 — Regresión Logística

Modelo lineal que estima probabilidades de clase mediante la función sigmoide (o softmax en multiclase). Es el modelo más interpretable de todos — sus coeficientes indican directamente la influencia de cada variable. Sirve como baseline lineal para comparar contra los modelos no lineales.

In [ ]:
from sklearn.linear_model import LogisticRegression

# también requiere escalado — las regularizaciones L1/L2 son sensibles a la magnitud
lr_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('lr', LogisticRegression(
        multi_class='multinomial', # softmax para 3 clases
        solver='lbfgs',
        max_iter=1000,
        class_weight='balanced',
        random_state=11
    ))
])

lr_pipeline.fit(X_train, y_train)
y_pred_lr = lr_pipeline.predict(X_test)

print("=== Regresión Logística ===")
print(classification_report(y_test, y_pred_lr, target_names=le_target_draft.classes_))


In [ ]:
cm_lr = confusion_matrix(y_test, y_pred_lr)
disp_lr = ConfusionMatrixDisplay(cm_lr, display_labels=le_target_draft.classes_)
disp_lr.plot(cmap='Blues')
plt.title('Matriz de Confusión — Regresión Logística')
plt.tight_layout()
plt.show()


## Comparativa final — todos los modelos

Resumen de los 6 modelos supervisados (XGBoost base, XGBoost RandomizedSearch, LightGBM, LightGBM top15, Random Forest, SVM, Regresión Logística) + el no supervisado K-Means está en el notebook de arquetipos.

In [ ]:
# reconstruyo base para la comparativa
modelo_base_ref = XGBClassifier(
    objective='multi:softmax', num_class=3,
    n_estimators=100, max_depth=4, learning_rate=0.1,
    random_state=11, eval_metric='mlogloss'
)
pesos_base = compute_sample_weight(class_weight='balanced', y=y_train)
modelo_base_ref.fit(X_train, y_train, sample_weight=pesos_base)
y_pred_base = modelo_base_ref.predict(X_test)

modelos_todos = [
    ('Base XGB',        y_pred_base),
    ('XGB RndSearch',   y_pred_rxgb),
    ('LightGBM',        y_pred_lgbm),
    ('LGBM top15',      y_pred_top),
    ('Random Forest',   y_pred_rf),
    ('SVM RBF',         y_pred_svm),
    ('Log. Regression', y_pred_lr),
]

rows = []
for nombre, preds in modelos_todos:
    rep = classification_report(y_test, preds, target_names=le_target_draft.classes_, output_dict=True)
    rows.append({
        'modelo':    nombre,
        'macro_f1':  round(rep['macro avg']['f1-score'], 4),
        'f1_ND':     round(rep['ND']['f1-score'], 4),
        'f1_R1':     round(rep['R1']['f1-score'], 4),
        'f1_R2':     round(rep['R2']['f1-score'], 4),
    })

df_todos = pd.DataFrame(rows).sort_values('macro_f1', ascending=False)
print(df_todos.to_string(index=False))


In [ ]:
# gráfico comparativo final
fig, ax = plt.subplots(figsize=(13, 5))
fig.patch.set_facecolor('#0a1019')
ax.set_facecolor('#0a1019')

nombres = df_todos['modelo'].tolist()
x_pos = np.arange(len(nombres))
cols_m    = ['macro_f1', 'f1_R1', 'f1_R2']
colores_c = ['#f97316', '#A32D2D', '#854F0B']
etiq_c    = ['macro F1', 'F1 R1', 'F1 R2']
ancho_c   = 0.22

for i, (col, color, etiq) in enumerate(zip(cols_m, colores_c, etiq_c)):
    vals = df_todos[col].tolist()
    barras = ax.bar(x_pos + i * ancho_c, vals, ancho_c, label=etiq, color=color, alpha=0.85)
    for b in barras:
        ax.text(b.get_x() + b.get_width() / 2, b.get_height() + 0.01,
                f'{b.get_height():.2f}', ha='center', va='bottom', fontsize=7, color='white')

ax.set_xticks(x_pos + ancho_c)
ax.set_xticklabels(nombres, color='white', fontsize=9, rotation=15, ha='right')
ax.set_ylim(0, 1.0)
ax.set_ylabel('valor', color='white')
ax.set_title('Comparativa todos los modelos — predicción de ronda', color='white', fontsize=13, pad=12)
ax.tick_params(colors='white')
ax.legend(facecolor='#1a2535', labelcolor='white', fontsize=9)
for spine in ax.spines.values():
    spine.set_edgecolor('#ffffff22')
plt.tight_layout()
plt.show()


In [ ]:
os.makedirs('../../pkl/modelos', exist_ok=True)
os.makedirs('../../pkl/preprocesado', exist_ok=True)

joblib.dump(mejor_xgb,    '../../pkl/modelos/xgb_draft_randsearch.pkl')
joblib.dump(lgbm_model,   '../../pkl/modelos/lgbm_draft_balanceado.pkl')
joblib.dump(lgbm_top,     '../../pkl/modelos/lgbm_draft_top15.pkl')
joblib.dump(rf_model,     '../../pkl/modelos/rf_draft.pkl')
joblib.dump(svm_pipeline, '../../pkl/modelos/svm_draft.pkl')
joblib.dump(lr_pipeline,  '../../pkl/modelos/lr_draft.pkl')
joblib.dump(top15_vars,   '../../pkl/preprocesado/top15_vars_draft.pkl')
# el label encoder ya existe del primer modelo
# joblib.dump(le_target_draft, '../../pkl/preprocesado/le_target_draft.pkl')

print("Modelos guardados")
